<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Quantize Cosmos3-Super Text-to-Image to FP8

Cosmos3-Super Text-to-Image ships in two forms: the full-quality model and a **4-step
distilled** version tuned for fast, few-step generation. This notebook quantizes both to
FP8 — the base model first, then the distilled model.

The two are the *same* network; they are served with a different **sampler** (scheduler,
number of steps, guidance). Calibration replays the sampler each version actually uses, so
we quantize each one with its own sampler — spelled out in the distilled section below.

## 1. Prerequisites

Use a Linux machine with an NVIDIA GPU, model access on Hugging Face, and either
`uvx hf@latest auth login` or `HF_TOKEN` set. Quantizing the Super (32B) checkpoints
needs a GPU with enough memory to hold the model in bf16 plus the quantizers; Nano (8B)
is comfortable on a single 48 GB GPU.

You also need free disk for the Hugging Face checkpoint cache and for the FP8 output
(each output is roughly the size of the bf16 source). Point `HF_HOME` at a large volume
in the next step.

> **Headless servers:** if you see `libGL.so.1: cannot open shared object file` when the
> VAE loads, install the system graphics libraries:
>
> ```bash
> apt-get install -y libgl1 libglib2.0-0
> ```

## 2. Configure Paths and Environment

The defaults are relative to this `cosmos` checkout and use the CUDA 13 (or 12.8) Torch
backend depending on your system. Override any of these before running the cell:

```bash
export COSMOS3_QUANTIZE_VENV=/path/to/.venv-cosmos3-quantize
export COSMOS3_TORCH_BACKEND=cu130       # or cu128
export HF_HOME=/path/to/large/huggingface/cache
export OUTPUT_ROOT=/path/to/fp8/outputs
```

In [1]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'cookbooks').exists():
            return path
    return start


def configure_quantize_environment() -> None:
    global COSMOS_ROOT, QUANTIZE_ROOT, COSMOS3_QUANTIZE_VENV
    global COSMOS3_TORCH_BACKEND, OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    QUANTIZE_ROOT = COSMOS_ROOT / 'cookbooks' / 'cosmos3' / 'quantization'
    COSMOS3_QUANTIZE_VENV = Path(
        os.environ.get('COSMOS3_QUANTIZE_VENV', COSMOS_ROOT / '.venv-cosmos3-quantize')
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get('COSMOS3_TORCH_BACKEND', 'cu130')
    OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', QUANTIZE_ROOT / 'outputs')).resolve()

    os.environ['COSMOS3_QUANTIZE_VENV'] = str(COSMOS3_QUANTIZE_VENV)
    os.environ['COSMOS3_TORCH_BACKEND'] = COSMOS3_TORCH_BACKEND
    os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    os.environ.setdefault('UV_LINK_MODE', 'copy')
    os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
    # Keep the datasets cache writable and separate from the shared hub lock dir.
    os.environ.setdefault('HF_DATASETS_CACHE', str(Path(os.environ['HF_HOME']) / 'datasets'))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configure_quantize_environment()
print('COSMOS_ROOT   :', COSMOS_ROOT)
print('QUANTIZE_ROOT :', QUANTIZE_ROOT)
print('OUTPUT_ROOT   :', OUTPUT_ROOT)
print('venv          :', COSMOS3_QUANTIZE_VENV)

COSMOS_ROOT   : <COSMOS>
QUANTIZE_ROOT : <COSMOS>/cookbooks/cosmos3/quantization
OUTPUT_ROOT   : <COSMOS>/cookbooks/cosmos3/quantization/outputs
venv          : <COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize


## 3. Install Dependencies

This creates a dedicated virtual environment with PyTorch, NVIDIA TensorRT Model
Optimizer (ModelOpt — the FP8 quantization engine), Diffusers (VAE + schedulers) and
Transformers (tokenizer), then registers a Jupyter kernel for it. Run this once.

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo 'uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/'
  exit 1
fi

export UV_LINK_MODE="${UV_LINK_MODE:-copy}"
uv venv "$COSMOS3_QUANTIZE_VENV" --python 3.13 --seed --managed-python --allow-existing
source "$COSMOS3_QUANTIZE_VENV/bin/activate"
uv pip install --torch-backend="$COSMOS3_TORCH_BACKEND" \
  "diffusers @ git+https://github.com/huggingface/diffusers.git" \
  "nvidia-modelopt[torch]" \
  cosmos-framework \
  accelerate datasets huggingface_hub imageio imageio-ffmpeg ipykernel \
  numpy pillow safetensors torch torchvision transformers

"$COSMOS3_QUANTIZE_VENV/bin/python" -m ipykernel install --user \
  --name cosmos3-quantize \
  --display-name "Cosmos3 Quantize (Python 3.13)"

echo
echo "Installed into: $COSMOS3_QUANTIZE_VENV"
echo "Next: switch this notebook kernel to: Cosmos3 Quantize (Python 3.13)"

## 4. Select the Quantization Kernel

The install cell registers the `Cosmos3 Quantize (Python 3.13)` Jupyter kernel.

**Switch this notebook to that kernel**, then run the restore cell below before
continuing. It can take a moment for a new kernel to appear in the notebook interface.

In [3]:
# Run this cell immediately after switching to the Cosmos3 Quantize kernel.
# It restores the same paths and cache settings as the Configure cell above.
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'cookbooks').exists():
            return path
    return start


def configure_quantize_environment() -> None:
    global COSMOS_ROOT, QUANTIZE_ROOT, COSMOS3_QUANTIZE_VENV
    global COSMOS3_TORCH_BACKEND, OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    QUANTIZE_ROOT = COSMOS_ROOT / 'cookbooks' / 'cosmos3' / 'quantization'
    COSMOS3_QUANTIZE_VENV = Path(
        os.environ.get('COSMOS3_QUANTIZE_VENV', COSMOS_ROOT / '.venv-cosmos3-quantize')
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get('COSMOS3_TORCH_BACKEND', 'cu130')
    OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', QUANTIZE_ROOT / 'outputs')).resolve()

    os.environ['COSMOS3_QUANTIZE_VENV'] = str(COSMOS3_QUANTIZE_VENV)
    os.environ['COSMOS3_TORCH_BACKEND'] = COSMOS3_TORCH_BACKEND
    os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    os.environ.setdefault('UV_LINK_MODE', 'copy')
    os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
    # Keep the datasets cache writable and separate from the shared hub lock dir.
    os.environ.setdefault('HF_DATASETS_CACHE', str(Path(os.environ['HF_HOME']) / 'datasets'))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configure_quantize_environment()
print('environment restored — OUTPUT_ROOT:', OUTPUT_ROOT)

environment restored — OUTPUT_ROOT: <COSMOS>/cookbooks/cosmos3/quantization/outputs


## 5. Verify GPU and Python Environment

Confirm the kernel sees a GPU and the quantization engine imports cleanly.

In [4]:
import torch, modelopt
print('torch          :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
print('ModelOpt       :', modelopt.__version__)

torch          : 2.13.0+cu130
CUDA available : True
GPU            : NVIDIA B200
ModelOpt       : 0.45.0


## 6. Load the Quantization Toolkit

The cookbook ships the Cosmos3 model and the complete FP8 recipe in `quantization/src/`:
loading, calibration, and export. `quantize_fp8_checkpoint(...)` runs the whole
pipeline for one checkpoint — load the model, **calibrate** it by replaying real
denoising so the quantizer sees representative activations, quantize to FP8, and write a
drop-in checkpoint you can serve.

The two helpers below wrap that call (resolving the Hugging Face checkpoint) and print a
short summary of what was produced.

In [ ]:
import sys
sys.path.insert(0, str(QUANTIZE_ROOT))

from huggingface_hub import snapshot_download
from safetensors import safe_open
import json
import src
from src import (quantize_fp8_checkpoint,
                 SHAPE_VIDEO, SHAPE_IMAGE, SHAPE_VIDEO_DEMO, SHAPE_IMAGE_DEMO,
                 SAMPLER_VIDEO_BASE, SAMPLER_IMAGE_BASE, SAMPLER_DISTILLED)

# DEMO=1 (default): one calibration prompt at a small shape, so a run takes minutes.
# Set DEMO=0 for the production shape and 8 calibration prompts (the shipped recipe).
DEMO = os.environ.get('DEMO', '1') == '1'
NUM_SAMPLES = 1 if DEMO else 8
print(f'DEMO={DEMO}  (NUM_SAMPLES={NUM_SAMPLES})')


def inspect_checkpoint(output_dir):
    """Print a short, human-readable summary of an FP8 checkpoint."""
    output_dir = Path(output_dir)
    tdir = output_dir / 'transformer'
    n_fp8 = n_scale = 0
    example = []
    for shard in sorted(tdir.glob('*.safetensors')):
        with safe_open(str(shard), framework='pt') as h:
            for k in h.keys():
                if k.endswith(('.input_scale', '.weight_scale')):
                    n_scale += 1
                    if k.endswith('.input_scale') and len(example) < 3:
                        example.append((k, float(h.get_tensor(k).reshape(-1)[0])))
                elif k.endswith('.weight') and h.get_slice(k).get_dtype() == 'F8_E4M3':
                    n_fp8 += 1
    qcfg = json.load(open(output_dir / 'hf_quant_config.json'))
    print(f'FP8 checkpoint: {output_dir}')
    print(f"  quant_algo   : {qcfg.get('quant_algo')}  (method={qcfg.get('quant_method')})")
    print(f'  FP8 weights  : {n_fp8}')
    print(f'  scale tensors: {n_scale}')
    for k, v in example:
        print(f'    e.g. {k} = {v:.6g}')

<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEMO=True  (NUM_SAMPLES=1)


<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize/lib/python3.13/site-packages/modelopt/torch/__init__.py:51: UserWarning: transformers 5.14.1 is not tested with current version of modelopt and may cause issues. Please install recommended version with `pip install -U nvidia-modelopt[hf]` if working with HF models.
  _warnings.warn(


## Cosmos3-Super Text-to-Image (Base)

The full-quality text-to-image model. It is served with a 50-step UniPC sampler, so that
is what we calibrate.

### Quantize

In [ ]:
model_name_or_path = os.environ.get('C3_T2I_DIR', 'nvidia/Cosmos3-Super-Text2Image')
output_dir = OUTPUT_ROOT / 'super-t2i-fp8'

quantize_fp8_checkpoint(
    model_name_or_path=model_name_or_path, output_dir=output_dir,
    profile='t2i', sampler=SAMPLER_IMAGE_BASE, shape=SHAPE_IMAGE_DEMO if DEMO else SHAPE_IMAGE,
    num_samples=NUM_SAMPLES,
)

### Inspect the FP8 Checkpoint

A quick look at the drop-in checkpoint just written: the quantization method, how many
weights were converted to FP8, and the per-tensor scales that accompany them.

In [7]:
inspect_checkpoint(OUTPUT_ROOT / 'super-t2i-fp8')

FP8 checkpoint: <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-t2i-fp8
  quant_algo   : FP8  (method=modelopt)
  FP8 weights  : 896
  scale tensors: 1792
    e.g. layers.0.mlp.down_proj.input_scale = 0.0245536
    e.g. layers.0.mlp.gate_proj.input_scale = 0.00247628
    e.g. layers.0.mlp.up_proj.input_scale = 0.00247628


## Cosmos3-Super Text-to-Image — 4-Step Distilled

The distilled model is the **same network** as the base, but it is served in just 4 steps.
The only thing that changes for quantization is the **sampler** — so calibration replays
the 4-step schedule instead of the 50-step one:

| | scheduler | steps | guidance |
|---|---|---|---|
| base | UniPC | 50 | 6.0 |
| distilled | FlowMatchEuler (4 fixed steps) | 4 | 1.0 (guidance off) |

The recipe reads the correct scheduler and step schedule from each checkpoint, so from
your side the call below is identical to the base — only `sampler=SAMPLER_DISTILLED`
changes.

### Quantize

In [ ]:
model_name_or_path = os.environ.get('C3_T2I_4STEP_DIR', 'nvidia/Cosmos3-Super-Text2Image-4Step')
output_dir = OUTPUT_ROOT / 'super-t2i-distilled-fp8'

quantize_fp8_checkpoint(
    model_name_or_path=model_name_or_path, output_dir=output_dir,
    profile='t2i', sampler=SAMPLER_DISTILLED, shape=SHAPE_IMAGE_DEMO if DEMO else SHAPE_IMAGE,
    num_samples=NUM_SAMPLES,
)

### Inspect the FP8 Checkpoint

A quick look at the drop-in checkpoint just written: the quantization method, how many
weights were converted to FP8, and the per-tensor scales that accompany them.

In [9]:
inspect_checkpoint(OUTPUT_ROOT / 'super-t2i-distilled-fp8')

FP8 checkpoint: <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-t2i-distilled-fp8
  quant_algo   : FP8  (method=modelopt)
  FP8 weights  : 896
  scale tensors: 1792
    e.g. layers.0.mlp.down_proj.input_scale = 0.0245536
    e.g. layers.0.mlp.gate_proj.input_scale = 0.00247628
    e.g. layers.0.mlp.up_proj.input_scale = 0.00247628
